# 05 — Clustering: perfiles de personal ESPOL

**Objetivo:** encontrar agrupaciones de personas con características similares a partir de
`data/modeling/X_modelado.csv`, evaluando distintos algoritmos y números de clusters, y
seleccionar una solución final interpretable y útil para el objetivo de la tesis
(apoyo a la asignación de tareas, conformación de comisiones/equipos, planificación).

> **Actualizado 2026-09-08 (DEC-007):** `X_modelado.csv` ahora incluye las features de
> trayectoria de `notebooks/04_trayectorias/04_trayectorias.ipynb` (tramos de rol,
> transiciones AA↔DD, antigüedad en la categoría de rol actual — DEC-006), y se corrigió
> el bug de vigencia de DEC-005. Ver sección 8 para la interpretación actualizada de los
> 5 perfiles.

**Entradas (ya preparadas en la etapa 04, no se repiten aquí):**

- `data/modeling/X_modelado.csv` — matriz numérica lista para modelar (2213 × 141), sin nulos ni infinitos.
- `data/modeling/personas_modelado.csv` — relación fila ↔ `IDPERSONA`.
- `data/modeling/feature_names_modelado.csv` — nombres de las 141 columnas finales y su origen.
- `data/modeling/preprocessing_summary.csv` — resumen de imputación/transformación/escalamiento por variable original.
- `data/features/dataset_personas_features.csv` — features **originales/interpretables** (85 variables), usadas
  únicamente para caracterizar los clusters, nunca para el clustering en sí.
- `data/trayectorias/features_trayectoria_persona.csv` — features de trayectoria, fusionadas en la etapa 04
  y usadas también para caracterizar los clusters (sección 7).

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
)
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

ROOT = Path.cwd().parents[1] if (Path.cwd().name == "06_clustering") else Path.cwd()
DATA_MODELING = ROOT / "data" / "modeling"
DATA_FEATURES = ROOT / "data" / "features"
DATA_CLUSTERING = ROOT / "data" / "clustering"
DATA_CLUSTERING.mkdir(parents=True, exist_ok=True)

print("Directorio de trabajo:", ROOT)
print("Salidas de clustering en:", DATA_CLUSTERING)


## 1. Carga y verificación de datos

In [ ]:
X_df = pd.read_csv(DATA_MODELING / "X_modelado.csv")
personas = pd.read_csv(DATA_MODELING / "personas_modelado.csv")
feature_names = pd.read_csv(DATA_MODELING / "feature_names_modelado.csv")
preprocessing_summary = pd.read_csv(DATA_MODELING / "preprocessing_summary.csv")

print("X_modelado:", X_df.shape)
print("personas_modelado:", personas.shape)
print("feature_names_modelado:", feature_names.shape)


In [ ]:
# Verificaciones básicas de integridad
n_nulos = int(X_df.isnull().sum().sum())
n_infinitos = int(np.isinf(X_df.to_numpy()).sum())

assert X_df.shape[0] == personas.shape[0], "Filas de X_modelado y personas_modelado no coinciden"
assert X_df.shape[1] == feature_names.shape[0], "Número de columnas no coincide con feature_names_modelado"
assert list(personas["INDICE_X_MODELADO"]) == list(range(len(personas))), \
    "El índice de personas_modelado no corresponde 1:1 con las filas de X_modelado"
assert personas["IDPERSONA"].is_unique, "Hay IDPERSONA duplicados en personas_modelado"

print(f"Personas               : {X_df.shape[0]}")
print(f"Features               : {X_df.shape[1]}")
print(f"Nulos totales          : {n_nulos}")
print(f"Infinitos totales      : {n_infinitos}")
print(f"IDPERSONA únicos       : {personas['IDPERSONA'].nunique()}")
print(f"Correspondencia fila<->IDPERSONA verificada: OK")


In [ ]:
# Distribución general de la matriz (ya escalada/codificada en la etapa 04)
X = X_df.to_numpy()

resumen_valores = pd.Series(X.ravel()).describe()
print(resumen_valores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X.ravel(), bins=80, color="#4C72B0")
axes[0].set_title("Distribución de todos los valores de X_modelado")
axes[0].set_xlabel("valor")
axes[0].set_ylabel("frecuencia")

col_std = X_df.std().sort_values(ascending=False)
axes[1].hist(col_std, bins=30, color="#55A868")
axes[1].set_title("Desviación estándar por columna")
axes[1].set_xlabel("std")
axes[1].set_ylabel("num. columnas")

plt.tight_layout()
plt.show()


**Nota sobre la naturaleza mixta de `X_modelado`:** según `preprocessing_summary.csv`, las 79
variables numéricas (y la ordinal `NIVEL_ACADEMICO_MAXIMO_ORD`) fueron estandarizadas con
`StandardScaler` (media 0, varianza 1), mientras que las variables categóricas codificadas con
one-hot (`nom__...`) y las binarias (`bin__...`) quedaron en escala 0/1 **sin estandarizar**. Esto
es una decisión ya tomada en la etapa 04 y no se modifica aquí, pero implica que, en un clustering
basado en distancia euclídea (K-Means, jerárquico), las variables numéricas estandarizadas
dominan más la distancia que las variables dummy 0/1. Se documenta como una característica/limitación
conocida de la representación utilizada, relevante al interpretar los resultados.


## 2. Selección del número de clusters (K) — K-Means

Se evalúan varios valores de K con K-Means (`n_init=10`, `random_state` fijo para esta primera
pasada) y se registran métricas de calidad y tamaños de cluster. La estabilidad ante distintas
semillas se evalúa por separado en la sección 3.


In [ ]:
K_RANGE = range(2, 13)

filas_k = []
labels_por_k = {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    labels_por_k[k] = labels
    sizes = pd.Series(labels).value_counts()
    filas_k.append({
        "K": k,
        "SILHOUETTE": silhouette_score(X, labels),
        "CALINSKI_HARABASZ": calinski_harabasz_score(X, labels),
        "DAVIES_BOULDIN": davies_bouldin_score(X, labels),
        "MIN_CLUSTER_SIZE": int(sizes.min()),
        "MAX_CLUSTER_SIZE": int(sizes.max()),
    })

evaluacion_k = pd.DataFrame(filas_k)
evaluacion_k.to_csv(DATA_CLUSTERING / "evaluacion_k.csv", index=False)
evaluacion_k


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(evaluacion_k["K"], evaluacion_k["SILHOUETTE"], marker="o")
axes[0, 0].set_title("Silhouette (mayor es mejor)")
axes[0, 0].set_xlabel("K")

axes[0, 1].plot(evaluacion_k["K"], evaluacion_k["CALINSKI_HARABASZ"], marker="o", color="#55A868")
axes[0, 1].set_title("Calinski-Harabasz (mayor es mejor)")
axes[0, 1].set_xlabel("K")

axes[1, 0].plot(evaluacion_k["K"], evaluacion_k["DAVIES_BOULDIN"], marker="o", color="#C44E52")
axes[1, 0].set_title("Davies-Bouldin (menor es mejor)")
axes[1, 0].set_xlabel("K")

axes[1, 1].plot(evaluacion_k["K"], evaluacion_k["MIN_CLUSTER_SIZE"], marker="o", label="mínimo")
axes[1, 1].plot(evaluacion_k["K"], evaluacion_k["MAX_CLUSTER_SIZE"], marker="o", label="máximo")
axes[1, 1].set_title("Tamaño de clusters")
axes[1, 1].set_xlabel("K")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


**Lectura de las métricas de K-Means:**

- El **Silhouette** es máximo en `K=2` y decrece de forma monótona después. Esto es esperable en una
  matriz de 100 dimensiones con variables dummy: una partición muy gruesa maximiza la separación
  promedio, pero no necesariamente aporta perfiles útiles para la tesis (ver más abajo qué separa
  a K=2).
- **Calinski-Harabasz** sigue el mismo patrón decreciente — también favorece soluciones gruesas y no
  es, por sí solo, un criterio suficiente.
- **Davies-Bouldin** (menor es mejor) es más plano entre K=4 y K=9, sin un mínimo claramente dominante.
- Los **tamaños de cluster** se mantienen razonablemente balanceados (ningún cluster
  extremadamente pequeño) hasta aproximadamente K=7-8; a partir de K=9 empiezan a aparecer clusters
  de menos de 100 personas.

Ninguna métrica por sí sola debe decidir K (así lo indica también la consigna del proyecto). Antes
de decidir, se revisa qué representa realmente la partición en K=2, y se evalúa la **estabilidad**
de las particiones (sección 3) y su **interpretabilidad** (sección 6/7).


In [ ]:
# ¿Qué separa a la partición más gruesa (K=2)?
feat_orig = pd.read_csv(DATA_FEATURES / "dataset_personas_features.csv")

tmp = personas[["IDPERSONA"]].copy()
tmp["CLUSTER_K2"] = labels_por_k[2]
tmp = tmp.merge(feat_orig[["IDPERSONA", "TIPOEMPLEADO_ACTUAL_DESC"]], on="IDPERSONA", how="left")

print(pd.crosstab(tmp["CLUSTER_K2"], tmp["TIPOEMPLEADO_ACTUAL_DESC"]))


K=2 no separa de forma limpia personal docente de administrativo (ambos tipos aparecen en los dos
clusters); el corte principal está dominado por el volumen general de actividad/antigüedad. Es una
partición demasiado agregada para el objetivo de la tesis (perfiles multidimensionales útiles para
asignación de tareas), por lo que **no se selecciona únicamente por tener el mayor Silhouette**.


## 3. Estabilidad de K-Means

K-Means depende de la inicialización aleatoria. Para cada K se ejecuta el algoritmo con varias
semillas distintas y se mide el **Adjusted Rand Index (ARI)** entre cada par de particiones
resultantes. Un ARI cercano a 1 indica que, independientemente de la semilla, el algoritmo converge
esencialmente a la misma partición (solución estable); valores bajos indican que la solución depende
demasiado del azar de inicialización para ese K.


In [ ]:
SEEDS = [0, 1, 2, 3, 4, 42, 100, 123]

filas_estab = []
for k in K_RANGE:
    labelings = [
        KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(X)
        for s in SEEDS
    ]
    aris = [
        adjusted_rand_score(labelings[i], labelings[j])
        for i in range(len(labelings))
        for j in range(i + 1, len(labelings))
    ]
    aris = np.array(aris)
    filas_estab.append({
        "K": k,
        "ARI_MEAN": aris.mean(),
        "ARI_STD": aris.std(),
        "ARI_MIN": aris.min(),
    })

estabilidad_k = pd.DataFrame(filas_estab)
estabilidad_k


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(
    estabilidad_k["K"], estabilidad_k["ARI_MEAN"], yerr=estabilidad_k["ARI_STD"],
    marker="o", capsize=3,
)
ax.axhline(0.9, color="gray", linestyle="--", linewidth=1, label="ARI = 0.90")
ax.set_xlabel("K")
ax.set_ylabel("ARI promedio entre semillas")
ax.set_title("Estabilidad de K-Means ante distintas semillas")
ax.legend()
plt.tight_layout()
plt.show()


**Lectura de estabilidad:** las particiones son muy estables (ARI ≥ 0.94) entre K=2 y K=7, con un
máximo local notable en **K=5** (ARI promedio ≈ 0.99). A partir de K=8 la estabilidad cae de forma
apreciable (ARI < 0.87 y con mayor dispersión), señal de que en ese rango K-Means empieza a producir
particiones distintas según la inicialización — es decir, la estructura de los datos ya no sostiene
con claridad ese número de grupos. Esto descarta, en la práctica, valores de K ≥ 8 para la solución
final, y refuerza a K=5 como un punto interesante dentro del rango estable.


## 4. Clustering jerárquico (Agglomerative)

Como segundo enfoque se evalúa clustering aglomerativo con enlace `ward` (el más comparable a
K-Means porque también minimiza varianza intra-cluster) y distancia euclídea, para el mismo rango
de K identificado como razonable (K=3 a K=7). No se prueban decenas de combinaciones de
enlace/distancia — solo configuraciones razonables — y el dendrograma se usa únicamente como
herramienta exploratoria, no como criterio de corte definitivo.


In [ ]:
# Dendrograma exploratorio (truncado; ward/euclídea sobre X completo)
Z = linkage(X, method="ward")

fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True, ax=ax)
ax.set_title("Dendrograma (ward, truncado a los últimos 30 nodos) — solo exploratorio")
ax.set_xlabel("tamaño de cluster (o índice) en cada nodo")
ax.set_ylabel("distancia (ward)")
plt.tight_layout()
plt.show()


In [ ]:
filas_agg = []
labels_agg_por_k = {}

for k in [3, 4, 5, 6, 7]:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels_agg = agg.fit_predict(X)
    labels_agg_por_k[k] = labels_agg
    sizes = pd.Series(labels_agg).value_counts()
    ari_vs_kmeans = adjusted_rand_score(labels_por_k[k], labels_agg)
    filas_agg.append({
        "K": k,
        "SILHOUETTE": silhouette_score(X, labels_agg),
        "CALINSKI_HARABASZ": calinski_harabasz_score(X, labels_agg),
        "DAVIES_BOULDIN": davies_bouldin_score(X, labels_agg),
        "MIN_CLUSTER_SIZE": int(sizes.min()),
        "MAX_CLUSTER_SIZE": int(sizes.max()),
        "ARI_VS_KMEANS": ari_vs_kmeans,
    })

comparacion_agg = pd.DataFrame(filas_agg)
comparacion_agg


**Comparación jerárquico vs. K-Means:** para todos los K evaluados, K-Means iguala o supera a
Agglomerative(ward) en Silhouette, Calinski-Harabasz y Davies-Bouldin. El ARI entre ambas
soluciones para el mismo K se mantiene entre ~0.5 y ~0.66 — es decir, **coinciden de forma
sustancial pero no idéntica**: hay acuerdo estructural razonable entre dos algoritmos con criterios
de optimización distintos, lo que da algo de confianza en que la estructura detectada no es un
artefacto de un solo algoritmo. K-Means se usa como referencia principal por su mejor desempeño en
las tres métricas; el jerárquico se usa como validación cruzada del enfoque, no como candidato
final independiente.


## 5. DBSCAN / HDBSCAN

Se evalúan métodos basados en densidad **únicamente si tienen sentido para esta matriz**: 100
dimensiones, con una mezcla de variables numéricas estandarizadas y variables dummy 0/1 dispersas
(one-hot y binarias). En alta dimensionalidad, las distancias tienden a concentrarse y la densidad
deja de ser un concepto bien definido ("curse of dimensionality"), por lo que antes de forzar el uso
de estos métodos se hace una verificación empírica rápida.


In [ ]:
# Distancia al 10º vecino más cercano, para orientar la elección de eps en DBSCAN
nn = NearestNeighbors(n_neighbors=10).fit(X)
dist_10nn, _ = nn.kneighbors(X)
k_dist = np.sort(dist_10nn[:, -1])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(k_dist)
ax.set_title("Distancia al 10º vecino más cercano (ordenada) — orientación para eps de DBSCAN")
ax.set_xlabel("puntos, ordenados por distancia")
ax.set_ylabel("distancia al 10º vecino")
plt.tight_layout()
plt.show()

print(pd.Series(k_dist).describe())


In [ ]:
filas_dbscan = []
for eps in [3, 4, 5, 6, 8, 10]:
    for min_samples in [5, 10]:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int((labels == -1).sum())
        filas_dbscan.append({
            "EPS": eps, "MIN_SAMPLES": min_samples,
            "N_CLUSTERS": n_clusters, "N_NOISE": n_noise,
            "PCT_NOISE": round(100 * n_noise / len(labels), 1),
        })

dbscan_trials = pd.DataFrame(filas_dbscan)
dbscan_trials


In [ ]:
filas_hdbscan = []
for mcs in [20, 30, 50, 80, 100]:
    labels = HDBSCAN(min_cluster_size=mcs).fit_predict(X)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    filas_hdbscan.append({
        "MIN_CLUSTER_SIZE": mcs,
        "N_CLUSTERS": n_clusters, "N_NOISE": n_noise,
        "PCT_NOISE": round(100 * n_noise / len(labels), 1),
    })

hdbscan_trials = pd.DataFrame(filas_hdbscan)
hdbscan_trials


**Conclusión DBSCAN/HDBSCAN — no se usan como solución final:**

- Con `eps` bajo (3-4), DBSCAN deja entre 75% y 93% de las personas como *ruido*, formando solo 2-5
  clusters diminutos.
- Al aumentar `eps` (8-10) para reducir el ruido, el algoritmo colapsa a **un único cluster gigante**
  que contiene prácticamente a todos (0.9%-9.8% de ruido, 1 solo cluster) — no hay un rango
  intermedio de `eps` que produzca varios clusters sustanciales con poco ruido.
- HDBSCAN muestra el mismo patrón: con `min_cluster_size` bajo encuentra 2 clusters muy pequeños
  (decenas de personas) dejando >90% como ruido; con `min_cluster_size` ≥ 50 no encuentra ningún
  cluster (100% ruido).

Este comportamiento es el esperado en una matriz de 100 dimensiones con variables dummy dispersas:
no existe una noción de densidad bien diferenciada que separe grupos densos de regiones vacías. Tal
como indica la consigna del proyecto, **se documenta esta limitación y no se fuerza el uso de
métodos basados en densidad** para la solución final; K-Means y el jerárquico (ambos basados en
distancia/varianza, no en densidad) son más adecuados para esta representación.


## 6. Selección del modelo final

Se resumen los criterios evaluados para los K candidatos (aquellos con estabilidad alta, K=3 a K=7)
bajo K-Means, y se comparan cualitativamente contra el jerárquico y contra la partición trivial K=2.

Criterios considerados conjuntamente (ninguno de forma aislada):

1. **Calidad estadística** (Silhouette, Calinski-Harabasz, Davies-Bouldin).
2. **Estabilidad** (ARI promedio entre semillas, sección 3).
3. **Tamaños razonables de cluster** (sin grupos extremadamente pequeños).
4. **Acuerdo con un algoritmo independiente** (ARI vs. Agglomerative, sección 4).
5. **Interpretabilidad / utilidad para el objetivo de la tesis** (se verifica a continuación con las
   features originales, antes de decidir).


In [ ]:
resumen_candidatos = (
    evaluacion_k[evaluacion_k["K"].between(3, 7)]
    .merge(estabilidad_k, on="K")
    .merge(comparacion_agg[["K", "ARI_VS_KMEANS"]], on="K")
)
resumen_candidatos


In [ ]:
# Vista previa de interpretabilidad: mediana de variables clave por cluster, para cada K candidato
feat_orig_idx = personas[["IDPERSONA"]].merge(feat_orig, on="IDPERSONA", how="left")

cols_preview = [
    "TOTAL_HORAS_DOCENCIA", "NUM_PUBLICACIONES", "NUM_PROYECTOS_INVESTIGACION",
    "ANIOS_EXPERIENCIA_DOCENTE", "ANIOS_EXPERIENCIA_ADMINISTRATIVO", "ANTIGUEDAD_EFECTIVA_ANIOS",
]

for k in [3, 4, 5, 6]:
    df_prev = feat_orig_idx.copy()
    df_prev["CLUSTER"] = labels_por_k[k]
    print(f"--- K={k} (tamaños: {df_prev['CLUSTER'].value_counts().sort_index().to_dict()}) ---")
    print(df_prev.groupby("CLUSTER")[cols_preview].median().round(2))
    print()


**Decisión — K-Means con K=5:**

> **Actualizado 2026-09-08 (DEC-007, reemplaza DEC-001):** recalculado tras corregir el bug de
> vigencia (DEC-005) y fusionar las features de trayectoria de `04_trayectorias.ipynb` a
> `X_modelado.csv` (ahora 141 columnas, antes 100). K=5 se mantiene como la mejor solución bajo
> los mismos criterios; cambian los números y, con ellos, la composición e interpretación de los
> clusters (sección 8).

- **Estabilidad:** K=5 sigue teniendo la estabilidad más alta de todo el rango evaluado (ARI
  promedio ≈ 0.97 entre semillas), muy por encima de K=8 en adelante (donde cae a 0.7-0.8).
- **Calidad estadística:** dentro del rango K=3-7 (una vez descartado K=2 por ser demasiado grueso),
  K=5 mantiene un Silhouette y Davies-Bouldin comparables a sus vecinos, sin ser el peor en ninguna
  métrica.
- **Acuerdo con el jerárquico:** K=5 sigue siendo el K con mayor ARI frente a Agglomerative(ward)
  del rango candidato, aunque el acuerdo bajó de ~0.66 (antes de las features de trayectoria) a
  ~0.56 — las nuevas features (en particular las de movilidad de rol) introducen una estructura que
  K-Means y el jerárquico capturan de forma algo más distinta entre sí que antes, sin dejar de
  coincidir en lo esencial.
- **Tamaños de cluster:** los 5 grupos quedan entre 209 y 644 personas (9.4%-29.1% de la
  población) — algo más desbalanceado que antes de agregar trayectoria, pero ningún grupo
  extremadamente pequeño ni dominante.
- **Interpretabilidad:** al revisar las medianas de variables originales y de trayectoria (sección
  7), K=5 sigue siendo el primer K que separa con claridad patrones sustantivos y distintos, y
  ahora aparece explícitamente un patrón que antes no se distinguía con nitidez: un grupo de alta
  **movilidad de rol** (transiciones administrativo↔docente repetidas), ver sección 8.

Por lo tanto, **no se elige K únicamente por el mejor Silhouette** (que sería K=2) ni se usa un
método basado en densidad (descartado en la sección 5). Se selecciona **K-Means, K=5,
`random_state=42`, `n_init=10`** como solución final, por ser la que mejor combina estabilidad,
calidad estadística competitiva, tamaños balanceados e interpretabilidad sustantiva para el objetivo
de la tesis. Esta decisión queda registrada en `context/DECISION_LOG.md` como DEC-007 (reemplaza
DEC-001).


In [ ]:
K_FINAL = 5

modelo_final = KMeans(n_clusters=K_FINAL, n_init=10, random_state=RANDOM_STATE)
cluster_labels = modelo_final.fit_predict(X)

print("Tamaños de cluster:")
print(pd.Series(cluster_labels).value_counts().sort_index())
print()
print(f"Silhouette         : {silhouette_score(X, cluster_labels):.4f}")
print(f"Calinski-Harabasz  : {calinski_harabasz_score(X, cluster_labels):.2f}")
print(f"Davies-Bouldin     : {davies_bouldin_score(X, cluster_labels):.4f}")


In [ ]:
# data/clustering/clusters_personas.csv — una fila por persona
clusters_personas = personas[["IDPERSONA"]].copy()
clusters_personas["CLUSTER"] = cluster_labels

assert clusters_personas["IDPERSONA"].is_unique
assert len(clusters_personas) == X_df.shape[0]

clusters_personas.to_csv(DATA_CLUSTERING / "clusters_personas.csv", index=False)
clusters_personas.head()


In [ ]:
# Modelo serializado
joblib.dump(modelo_final, DATA_CLUSTERING / "modelo_clustering.joblib")
print("Modelo guardado en", DATA_CLUSTERING / "modelo_clustering.joblib")


In [ ]:
# data/clustering/clustering_metrics.csv — métricas del modelo final seleccionado
ari_estab_final = estabilidad_k.loc[estabilidad_k["K"] == K_FINAL, "ARI_MEAN"].iloc[0]
ari_vs_agg_final = comparacion_agg.loc[comparacion_agg["K"] == K_FINAL, "ARI_VS_KMEANS"].iloc[0]

clustering_metrics = pd.DataFrame([{
    "MODELO": "KMeans",
    "K": K_FINAL,
    "N_PERSONAS": X_df.shape[0],
    "N_FEATURES": X_df.shape[1],
    "SILHOUETTE": silhouette_score(X, cluster_labels),
    "CALINSKI_HARABASZ": calinski_harabasz_score(X, cluster_labels),
    "DAVIES_BOULDIN": davies_bouldin_score(X, cluster_labels),
    "ARI_ESTABILIDAD_SEMILLAS": ari_estab_final,
    "ARI_VS_AGGLOMERATIVE": ari_vs_agg_final,
    "RANDOM_STATE": RANDOM_STATE,
    "N_INIT": 10,
}])
clustering_metrics.to_csv(DATA_CLUSTERING / "clustering_metrics.csv", index=False)
clustering_metrics


In [ ]:
# data/clustering/cluster_sizes.csv
cluster_sizes = (
    clusters_personas["CLUSTER"].value_counts().sort_index()
    .rename_axis("CLUSTER").reset_index(name="N_PERSONAS")
)
cluster_sizes["PCT_POBLACION"] = (100 * cluster_sizes["N_PERSONAS"] / len(clusters_personas)).round(2)
cluster_sizes.to_csv(DATA_CLUSTERING / "cluster_sizes.csv", index=False)
cluster_sizes


## 7. Caracterización de clusters

La caracterización se hace **con las features originales/interpretables** de
`data/features/dataset_personas_features.csv` (85 variables, en su unidad y escala natural)
más las features de trayectoria de `data/trayectorias/features_trayectoria_persona.csv`
(DEC-006), **no** con las 141 columnas transformadas de `X_modelado.csv` que se usaron
para el clustering. Se relaciona `IDPERSONA` + `CLUSTER` (de `clusters_personas.csv`) con
ambos datasets.

Cada variable original se trata según su tipo, tomado de `data/features/feature_dictionary.csv`
(las columnas de trayectoria no están ahí — se generaron después — y se declaran a mano
en `TIPO_TRAYECTORIA`):

- **Numéricas:** mediana por cluster vs. mediana global; `IMPORTANCE` = diferencia estandarizada
  (`|mediana_cluster - mediana_global| / std_global`), un tamaño de efecto simple.
- **Booleanas:** proporción de `True` por cluster vs. proporción global; `IMPORTANCE` = diferencia
  absoluta de proporciones.
- **Categóricas (incluye la ordinal `NIVEL_ACADEMICO_MAXIMO`):** se identifica la categoría más
  frecuente a nivel global y se compara qué proporción de cada cluster cae en esa categoría frente a
  la proporción global; adicionalmente se reporta la categoría más frecuente **dentro** de cada
  cluster (que puede diferir de la global), para la interpretación cualitativa.

`IDPERSONA` se excluye por ser un identificador, no una característica.


In [ ]:
feat_orig = pd.read_csv(DATA_FEATURES / "dataset_personas_features.csv")
feat_dict = pd.read_csv(DATA_FEATURES / "feature_dictionary.csv")

# Features de trayectoria (04_trayectorias.ipynb, DEC-006): no estaban en
# dataset_personas_features.csv/feature_dictionary.csv (generados antes de esa fase), se
# fusionan aqui solo para caracterizacion/interpretacion, con su tipo declarado a mano.
feat_trayectoria = pd.read_csv(ROOT / "data" / "trayectorias" / "features_trayectoria_persona.csv")
feat_orig = feat_orig.merge(feat_trayectoria, on="IDPERSONA", how="left")

TIPO_TRAYECTORIA = {
    "N_TRAMOS_ROL": "numerica", "N_CATEGORIAS_ROL_DISTINTAS": "numerica",
    "ANIOS_EN_CATEGORIA_ACTUAL": "numerica", "N_TRANSICIONES_ROL": "numerica",
    "N_TRANSICIONES_AA_A_DD": "numerica", "N_TRANSICIONES_DD_A_AA": "numerica",
    "ANIOS_DESDE_ULTIMA_TRANSICION": "numerica",
    "ES_TRAYECTORIA_ESTABLE": "booleana", "TUVO_TRANSICION_AA_A_DD": "booleana",
    "TUVO_TRANSICION_DD_A_AA": "booleana",
    "CATEGORIA_CARGO_ACTUAL": "categorica", "CATEGORIA_CARGO_PRIMERA": "categorica",
    "TIPOEMPLEADO_CATEGORIA_ACTUAL": "categorica",
    # DEC-011 (roles simultaneos, ver 04_trayectorias.ipynb / resolver_roles_simultaneos):
    "TUVO_ROL_ADICIONAL_SIMULTANEO": "booleana",
    "N_ROLES_ADICIONALES_SIMULTANEOS_DISTINTOS": "numerica",
    "CATEGORIA_ROL_ADICIONAL_MAS_RECIENTE": "categorica",
    # DEC-012 (funciones adicionales / registro_autoridades):
    "TUVO_FUNCION_ADICIONAL": "booleana",
    "N_FUNCIONES_ADICIONALES": "numerica",
    "N_CATEGORIAS_FUNCION_ADICIONAL_DISTINTAS": "numerica",
    "TUVO_SUBROGACION": "booleana",
    "N_SUBROGACIONES": "numerica",
    "TUVO_FUNCION_NO_COINCIDENTE_CON_CONTRATO": "booleana",
    "CATEGORIA_FUNCION_ADICIONAL_MAS_RECIENTE": "categorica",
    # DEC-016 (cargo/unidad del tramo estructural, no del ultimo contrato por fecha):
    "CARGO_ACTUAL_ESTRUCTURAL": "categorica", "UNIDAD_ACTUAL_ESTRUCTURAL": "categorica",
    "VIGENTE_TRAMO_ESTRUCTURAL": "booleana",
    # DEC-017 (fallback de presentacion cuando no hay tramo estructural vigente hoy):
    "CARGO_PUNTUAL_VIGENTE": "categorica", "UNIDAD_PUNTUAL_VIGENTE": "categorica",
    "CATEGORIA_PUNTUAL_VIGENTE": "categorica", "TIPOEMPLEADO_PUNTUAL_VIGENTE": "categorica",
}

df_carac = clusters_personas.merge(feat_orig, on="IDPERSONA", how="left")
assert df_carac["CLUSTER"].isnull().sum() == 0

tipo_por_feature = dict(zip(feat_dict["FEATURE"], feat_dict["TYPE"]))
tipo_por_feature.update(TIPO_TRAYECTORIA)
features_a_caracterizar = [c for c in feat_orig.columns if c != "IDPERSONA"]

clusters_ordenados = sorted(df_carac["CLUSTER"].unique())
filas_caract = []

for feature in features_a_caracterizar:
    tipo = tipo_por_feature.get(feature, "numerica")
    col = df_carac[feature]

    if tipo == "numerica":
        global_val = col.median()
        global_std = col.std()
        for c in clusters_ordenados:
            val_c = col[df_carac["CLUSTER"] == c].median()
            diff = val_c - global_val
            importance = abs(diff) / global_std if global_std and global_std > 0 else 0.0
            if diff > 0:
                signo = "muy por encima" if importance >= 0.8 else "por encima"
            elif diff < 0:
                signo = "muy por debajo" if importance >= 0.8 else "por debajo"
            else:
                signo = "en línea con"
            interpretacion = f"Mediana de {feature} {signo} del promedio institucional ({val_c:g} vs {global_val:g})."
            filas_caract.append(dict(
                CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                DIFFERENCE=diff, IMPORTANCE=importance, INTERPRETACION=interpretacion,
            ))

    elif tipo == "booleana":
        col_bool = col.astype("boolean")
        global_val = col_bool.mean(skipna=True)
        for c in clusters_ordenados:
            val_c = col_bool[df_carac["CLUSTER"] == c].mean(skipna=True)
            diff = val_c - global_val
            importance = abs(diff)
            interpretacion = (
                f"{val_c:.0%} del cluster cumple {feature}, frente a {global_val:.0%} a nivel global."
            )
            filas_caract.append(dict(
                CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                DIFFERENCE=diff, IMPORTANCE=importance, INTERPRETACION=interpretacion,
            ))

    elif tipo in ("categorica", "categorica_ordinal"):
        col_str = col.astype("string").fillna("SIN_DATO")
        vc_global = col_str.value_counts(normalize=True)
        global_mode, global_val = vc_global.index[0], vc_global.iloc[0]
        for c in clusters_ordenados:
            sub = col_str[df_carac["CLUSTER"] == c]
            vc_c = sub.value_counts(normalize=True)
            cluster_mode, cluster_mode_prop = vc_c.index[0], vc_c.iloc[0]
            val_c = vc_c.get(global_mode, 0.0)
            diff = val_c - global_val
            importance = abs(diff)
            interpretacion = (
                f"Categoría predominante en el cluster: '{cluster_mode}' ({cluster_mode_prop:.0%}); "
                f"a nivel global la más común es '{global_mode}' ({global_val:.0%})."
            )
            filas_caract.append(dict(
                CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                DIFFERENCE=diff, IMPORTANCE=importance, INTERPRETACION=interpretacion,
            ))
    # tipo "identificador" (no debería aparecer aquí) se ignora

cluster_characterization = (
    pd.DataFrame(filas_caract)
    .sort_values(["CLUSTER", "IMPORTANCE"], ascending=[True, False])
    .reset_index(drop=True)
)
cluster_characterization.to_csv(DATA_CLUSTERING / "cluster_characterization.csv", index=False)
print(cluster_characterization.shape)
cluster_characterization.head(10)


In [ ]:
# Top 8 variables más diferenciadoras por cluster (para revisión rápida / evidencia de nombres)
for c in clusters_ordenados:
    print(f"\n=== CLUSTER {c} (n={cluster_sizes.loc[cluster_sizes.CLUSTER==c, 'N_PERSONAS'].iloc[0]}) — variables más diferenciadoras ===")
    top = cluster_characterization[cluster_characterization["CLUSTER"] == c].head(8)
    for _, r in top.iterrows():
        print(f"  - {r['INTERPRETACION']}")


In [ ]:
# data/clustering/cluster_profiles.csv — resumen por cluster (formato ancho)
numericas = [f for f in features_a_caracterizar if tipo_por_feature.get(f) == "numerica"]
booleanas = [f for f in features_a_caracterizar if tipo_por_feature.get(f) == "booleana"]
categoricas = [f for f in features_a_caracterizar if tipo_por_feature.get(f) in ("categorica", "categorica_ordinal")]

perfiles = []
n_total = len(df_carac)
for c in clusters_ordenados:
    sub = df_carac[df_carac["CLUSTER"] == c]
    fila = {"CLUSTER": c, "N_PERSONAS": len(sub), "PCT_POBLACION": round(100 * len(sub) / n_total, 2)}
    for f in numericas:
        fila[f"{f}__MEDIA"] = sub[f].mean()
        fila[f"{f}__MEDIANA"] = sub[f].median()
    for f in booleanas:
        fila[f"{f}__PROPORCION"] = sub[f].astype("boolean").mean(skipna=True)
    for f in categoricas:
        vc = sub[f].astype("string").fillna("SIN_DATO").value_counts(normalize=True)
        fila[f"{f}__MODA"] = vc.index[0]
        fila[f"{f}__MODA_PROP"] = vc.iloc[0]
    perfiles.append(fila)

cluster_profiles = pd.DataFrame(perfiles)
cluster_profiles.to_csv(DATA_CLUSTERING / "cluster_profiles.csv", index=False)
print(cluster_profiles.shape)
cluster_profiles[["CLUSTER", "N_PERSONAS", "PCT_POBLACION"]]


## 8. Nombres de los perfiles

> **Actualizado 2026-09-08 (DEC-007):** recalculado con `X_modelado.csv` ampliado (features de
> trayectoria, DEC-006) y vigencia corregida (DEC-005). La numeración de clusters (0-4) es
> arbitraria (la asigna K-Means), no implica orden ni jerarquía.

Los nombres se proponen **después** de revisar la evidencia de la sección anterior (variables más
diferenciadoras por cluster, incluyendo ahora las de trayectoria), no antes. Se resume esa
evidencia por cluster:

- **Cluster 0** (271 personas, 12.2%): 97.4% docentes, con la mayor vigencia de los perfiles
  docentes (92.3% vs 57.7% institucional). Mayor producción académica: mediana 12.9 años de
  experiencia docente, 2868 horas de docencia, 11 proyectos de investigación y 17 publicaciones.
  Movilidad de rol moderada (35.1% tuvo transición administrativo→docente, 19.2% docente→administrativo,
  vs 16.8%/10.0% institucional).
  → **"Docente de alta producción académica e investigativa"**.

- **Cluster 1** (517 personas, 23.4%): 96.3% docentes, vigencia cercana a la mediana institucional
  (54.5% vs 57.7%). Carga docente moderada sin producción investigativa relevante (mediana 5 años
  de experiencia, 2419 horas de docencia, ~0 proyectos/publicaciones); rol actual predominante
  Docente No Titular Ocasional o Técnico Docente de Apoyo. Trayectoria mayormente estable (70.0%
  nunca tuvo una transición de rol).
  → **"Docente ocasional de carga media"**.

- **Cluster 2** (644 personas, 29.1%): 85.4% docentes pero con la menor vigencia de todos los
  perfiles (26.1% vs 57.7% institucional): personal de paso breve, con antigüedad mediana de solo
  1.4 años y prácticamente sin actividad registrada (0 horas de docencia, 0 proyectos). Rol actual
  predominante Técnico Docente de Apoyo. Trayectoria casi siempre estable (91.1%) simplemente por
  el poco tiempo de permanencia.
  → **"Personal técnico/docente de paso breve, mayormente no vigente"**.

- **Cluster 3** (209 personas, 9.4%): el perfil con mayor movilidad de rol, con diferencia: 76.1%
  tuvo alguna transición administrativo→docente y 63.2% docente→administrativo (vs 16.8%/10.0%
  institucional), con una mediana de 5 transiciones de rol detectadas sobre 14 tramos. También el
  de mayor antigüedad (mediana 25.5 años de calendario) y mayor experiencia combinada (17.6 años
  docente + 5.1 años administrativo), con alta vigencia actual (74.2%). Es el perfil que más
  directamente representa personal que ha atravesado la frontera administrativo/docente a lo largo
  de su carrera en ESPOL, en vez de permanecer en un solo lado — el patrón que originalmente
  motivó agregar el análisis de trayectorias a la tesis.
  → **"Trayectoria mixta docente-administrativo (alta movilidad de rol)"**.

- **Cluster 4** (572 personas, 25.9%): 96.0% administrativos, con la mayor experiencia
  administrativa (mediana 11.3 años vs 0.83 institucional), régimen LOSEP predominante (86%) y
  alta vigencia actual (73.8% vs 57.7% institucional). Rol actual predominante Profesional/Analista
  o Asistencia Secretarial.
  → **"Administrativo"**.

**Advertencia importante:** estos nombres son una síntesis interpretativa basada en los datos
disponibles, no una clasificación oficial ni una categoría administrativa de ESPOL. No implican un
juicio de valor sobre las personas de cada grupo, y no deben usarse como criterio automático de
decisiones sobre contratación, promoción o asignación sin la intervención y el criterio de las
unidades institucionales correspondientes (ver `context/00-contexto-permanente.md`).


In [ ]:
nombres_perfiles = {
    0: "Docente de alta producción académica e investigativa",
    1: "Docente ocasional de carga media",
    2: "Personal técnico/docente de paso breve, mayormente no vigente",
    3: "Trayectoria mixta docente-administrativo (alta movilidad de rol)",
    4: "Administrativo",
}

nombres_df = pd.DataFrame({
    "CLUSTER": list(nombres_perfiles.keys()),
    "NOMBRE_PERFIL": list(nombres_perfiles.values()),
})
cluster_sizes_nombrado = cluster_sizes.merge(nombres_df, on="CLUSTER")
cluster_sizes_nombrado


## 9. Visualizaciones

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
orden = cluster_sizes_nombrado.sort_values("CLUSTER")
bars = ax.bar(orden["CLUSTER"].astype(str), orden["N_PERSONAS"], color=sns.color_palette("Set2", len(orden)))
for bar, pct in zip(bars, orden["PCT_POBLACION"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5, f"{pct:.1f}%", ha="center")
ax.set_xlabel("Cluster")
ax.set_ylabel("N° de personas")
ax.set_title(f"Tamaño de los {K_FINAL} clusters finales (K-Means)")
plt.tight_layout()
plt.show()


### Dendrograma de los 5 perfiles finales (qué tan cerca están entre sí)

El dendrograma de la sección 4 (sobre puntos individuales, truncado) es solo exploratorio y no
está atado a la solución final. Aquí se construye un dendrograma **de los 5 centroides** del
modelo K-Means final (`modelo_final.cluster_centers_`, en el mismo espacio de 141 dimensiones de
`X_modelado`), con enlace `ward`. Esto responde directamente qué tan cerca/lejos está cada perfil
de los demás — p. ej. si "Administrativo" y "Trayectoria mixta" se parecen más entre sí que
"Administrativo" y "Docente de alta producción" — sin cambiar el modelo ni las asignaciones de
persona a cluster (siguen siendo las de K-Means, sección 6).


In [ ]:
centroides = modelo_final.cluster_centers_
etiquetas_centroides = [f"{c}: {nombres_perfiles[c]}\n(n={cluster_sizes_nombrado.loc[cluster_sizes_nombrado.CLUSTER==c, 'N_PERSONAS'].iloc[0]})"
                         for c in range(K_FINAL)]

Z_centroides = linkage(centroides, method="ward")

fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z_centroides, labels=etiquetas_centroides, ax=ax, leaf_rotation=0, leaf_font_size=9)
ax.set_title(f"Dendrograma de los {K_FINAL} perfiles finales (centroides, ward, espacio de X_modelado)")
ax.set_ylabel("distancia (ward) entre centroides")
plt.tight_layout()
plt.show()

# data/clustering/dendrograma_centroides.csv — matriz de enlace, para reconstruir el dendrograma
# en el dashboard sin recalcularlo (columnas estandar de scipy.linkage: ver su documentacion).
dendrograma_centroides = pd.DataFrame(
    Z_centroides, columns=["CLUSTER_A", "CLUSTER_B", "DISTANCIA", "N_OBSERVACIONES"]
)
dendrograma_centroides.to_csv(DATA_CLUSTERING / "dendrograma_centroides.csv", index=False)
dendrograma_centroides


### Visualización 2D (solo para inspección visual — no reemplaza la matriz de clustering)

Se proyecta `X_modelado` a 2 componentes con PCA **únicamente para poder graficar** los clusters en
un plano. El clustering en sí se realizó, y se selecciona, sobre las 100 dimensiones originales de
`X_modelado`; esta proyección 2D es solo una ayuda visual y pierde información — dos personas pueden
verse cercanas en el gráfico y no serlo en el espacio completo, o viceversa.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
var_explicada = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 6))
paleta = sns.color_palette("Set2", K_FINAL)
for c in clusters_ordenados:
    mask = cluster_labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=12, alpha=0.6, color=paleta[c], label=f"Cluster {c}")
ax.set_xlabel(f"PC1 ({var_explicada[0]:.1%} var. explicada)")
ax.set_ylabel(f"PC2 ({var_explicada[1]:.1%} var. explicada)")
ax.set_title("Clusters finales proyectados en 2D con PCA (solo visualización)")
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()

print(f"Varianza explicada por las 2 primeras componentes: {var_explicada.sum():.1%}")
print("(Baja cobertura esperable en 100 dimensiones: es una ayuda visual, no un resumen fiel de la matriz.)")


### Variables distintivas por cluster (heatmap de importancia estandarizada)

In [ ]:
# Top features numéricas más importantes en conjunto (unión de los top-6 de cada cluster)
top_por_cluster = (
    cluster_characterization[cluster_characterization["FEATURE"].isin(numericas)]
    .sort_values(["CLUSTER", "IMPORTANCE"], ascending=[True, False])
    .groupby("CLUSTER")
    .head(6)
)
features_heatmap = top_por_cluster["FEATURE"].unique().tolist()

pivot_diff = (
    cluster_characterization[cluster_characterization["FEATURE"].isin(features_heatmap)]
    .pivot(index="FEATURE", columns="CLUSTER", values="IMPORTANCE")
    .reindex(features_heatmap)
)

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(features_heatmap))))
sns.heatmap(pivot_diff, cmap="viridis", annot=True, fmt=".2f", ax=ax, cbar_kws={"label": "|diferencia| estandarizada"})
ax.set_title("Variables numéricas más distintivas por cluster (efecto estandarizado vs. global)")
ax.set_xlabel("Cluster")
plt.tight_layout()
plt.show()


In [ ]:
# Comparación de perfiles: TIPOEMPLEADO_ACTUAL_DESC y VIGENTE_ACTUALMENTE por cluster
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ct_tipo = pd.crosstab(df_carac["CLUSTER"], df_carac["TIPOEMPLEADO_ACTUAL_DESC"], normalize="index")
ct_tipo.plot(kind="bar", stacked=True, ax=axes[0], color=["#DD8452", "#4C72B0"])
axes[0].set_title("Tipo de empleado por cluster")
axes[0].set_ylabel("proporción")
axes[0].legend(title=None, fontsize=8)

vigencia = df_carac.groupby("CLUSTER")["VIGENTE_ACTUALMENTE"].apply(lambda s: s.astype("boolean").mean(skipna=True))
axes[1].bar(vigencia.index.astype(str), vigencia.values, color="#55A868")
axes[1].axhline(df_carac["VIGENTE_ACTUALMENTE"].astype("boolean").mean(skipna=True), color="gray", linestyle="--", label="promedio global")
axes[1].set_title("Proporción actualmente vigente por cluster")
axes[1].set_ylabel("proporción vigente")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 10. Resumen

**Decisión tomada:** K-Means con **K=5**, `random_state=42`, `n_init=10`, ajustado sobre las 141
columnas de `X_modelado.csv` (100 originales + features de trayectoria, DEC-006). Se descartó la
partición trivial K=2 (mejor Silhouette pero sin separación sustantiva), se validó cruzadamente
con clustering jerárquico (ward), y se documentó que DBSCAN/HDBSCAN no son adecuados para esta
matriz de alta dimensionalidad con variables dummy dispersas. Reemplaza DEC-001 como **DEC-007**
en `context/DECISION_LOG.md`: mismo K y mismo método, pero recalculado tras corregir el bug de
vigencia (DEC-005) y agregar trayectoria (DEC-006), lo que cambió la composición e interpretación
de los 5 perfiles (sección 8) — en particular, apareció un perfil de alta movilidad de rol
administrativo↔docente que antes no se distinguía.

**Archivos generados en `data/clustering/`:**

| Archivo | Contenido |
|---|---|
| `evaluacion_k.csv` | Métricas de K-Means para K=2..12 |
| `clusters_personas.csv` | `IDPERSONA` + `CLUSTER` final (una fila por persona) |
| `modelo_clustering.joblib` | Modelo K-Means final serializado |
| `clustering_metrics.csv` | Métricas del modelo final seleccionado |
| `cluster_sizes.csv` | Tamaño y % de población por cluster |
| `cluster_profiles.csv` | Resumen por cluster con features originales + trayectoria (media/mediana/proporciones/modas) |
| `cluster_characterization.csv` | Detalle `CLUSTER, FEATURE, VALUE_CLUSTER, VALUE_GLOBAL, DIFFERENCE, IMPORTANCE, INTERPRETACION` |
| `dendrograma_centroides.csv` | Matriz de enlace (ward) entre los 5 centroides finales, para la vista jerárquica del dashboard |

**Limitaciones y advertencias a tener presentes:**

- La matriz `X_modelado` mezcla variables numéricas estandarizadas con variables dummy 0/1 sin
  estandarizar; esto pondera más las variables numéricas en la distancia euclídea usada por K-Means
  y por el jerárquico.
- Los nombres de perfil de la sección 8 son una síntesis interpretativa, no una categoría
  institucional oficial ni una verdad absoluta sobre las personas — así lo establece el contexto
  permanente del proyecto.
- La proyección PCA de la sección 9 es solo una ayuda visual (baja varianza explicada en 2D dado que
  hay 141 dimensiones); no reemplaza ni resume fielmente la matriz usada para clustering.
- El dendrograma de centroides muestra cercanía entre los 5 *perfiles agregados*, no entre personas
  individuales — dos perfiles cercanos en el dendrograma pueden seguir teniendo personas muy
  distintas entre sí dentro de cada uno.
- El patrón de "encargos cortos" documentado en `04_trayectorias.ipynb` (sección 5) puede inflar el
  conteo de transiciones de algunas personas puntuales; no se filtró para esta fusión de features
  (queda como limitación conocida heredada de DEC-004).
- Esta solución no debe usarse para automatizar decisiones sensibles de contratación, promoción o
  asignación sin intervención humana.

**Siguiente paso sugerido:** regenerar `notebooks/08_dashboard/08_dashboard.ipynb` con estos
resultados actualizados (nombres/descripciones de perfil ya actualizados en `lib.py`) y agregar la
vista del dendrograma de centroides al dashboard Streamlit; opcionalmente, revisar si
`notebooks/07_embeddings/07_embeddings.ipynb` debe re-ejecutarse para comparar contra los nuevos
clusters.


## 11. Clustering jerárquico en 2 niveles: AA/DD primero, sub-perfiles después (DEC-008)

> **2026-09-08 (DEC-008, reemplaza DEC-007 como estructura final):** el usuario pidió explícitamente
> que los **dos grupos principales** de la visualización sean Administrativo y Docente (un hecho ya
> conocido del dato, `TIPOEMPLEADO_ACTUAL_DESC`, no algo que deba "descubrir" el clustering), y que
> dentro de cada uno se identifiquen sub-perfiles propios — en vez de un único K-Means global de 5
> clusters donde el tipo de empleado es una consecuencia y no una regla.

**Diseño:** en vez de clusterizar las 2213 personas juntas (secciones 1-10), se separa la población
en dos ramas por `TIPOEMPLEADO_ACTUAL_DESC` (dato institucional, no un resultado de clustering), y
se corre K-Means **por separado dentro de cada rama**, sobre las mismas 141 columnas de
`X_modelado.csv`, cada una con su propio K óptimo (mismos criterios de las secciones 2-6: estabilidad
entre semillas, calidad estadística, tamaños balanceados, interpretabilidad — sin repetir aquí la
comparación completa contra jerárquico/DBSCAN, que ya se hizo sobre la población conjunta y sus
conclusiones metodológicas (K-Means > jerárquico > densidad para esta matriz) siguen aplicando por
rama).

Esto **reemplaza como estructura final** la solución de K=5 global de DEC-007: los archivos de
`data/clustering/` se sobreescriben con esta versión de 2 niveles. Las secciones 1-10 se conservan
como documentación del proceso exploratorio (siguen siendo evidencia válida de por qué K-Means y no
DBSCAN/jerárquico, y de cómo se comporta la población conjunta), pero **ya no son la solución
vigente**.


In [ ]:
GRUPO_PRINCIPAL_COL = "TIPOEMPLEADO_ACTUAL_DESC"

grupo_principal = personas[["IDPERSONA"]].merge(
    feat_orig[["IDPERSONA", GRUPO_PRINCIPAL_COL]], on="IDPERSONA", how="left"
)[GRUPO_PRINCIPAL_COL]
# Las mismas 5 personas (0.23%) sin registro en historial_laboral (ver 05_preparacion_modelado,
# CATEGORICA_DESCONOCIDO) no tienen TIPOEMPLEADO_ACTUAL_DESC: no se les puede asignar una rama
# real sin inventar informacion, quedan fuera de esta clusterizacion de 2 niveles (CLUSTER=-1).
grupo_principal = grupo_principal.fillna("DESCONOCIDO").to_numpy()
print(pd.Series(grupo_principal).value_counts())

SEEDS_RAMA = [0, 1, 2, 42, 100]
K_RANGE_RAMA = range(2, 9)

evaluacion_por_rama = []
for rama in ["ADMINISTRATIVO", "DOCENTE"]:
    mask = grupo_principal == rama
    Xr = X[mask]
    for k in K_RANGE_RAMA:
        labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(Xr)
        sizes = pd.Series(labels).value_counts()
        labelings = [KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(Xr) for s in SEEDS_RAMA]
        aris = [adjusted_rand_score(labelings[i], labelings[j])
                for i in range(len(labelings)) for j in range(i + 1, len(labelings))]
        evaluacion_por_rama.append({
            "GRUPO_PRINCIPAL": rama, "K": k,
            "SILHOUETTE": silhouette_score(Xr, labels),
            "DAVIES_BOULDIN": davies_bouldin_score(Xr, labels),
            "MIN_CLUSTER_SIZE": int(sizes.min()), "MAX_CLUSTER_SIZE": int(sizes.max()),
            "ARI_ESTABILIDAD": float(np.mean(aris)),
        })

evaluacion_por_rama = pd.DataFrame(evaluacion_por_rama)
evaluacion_por_rama.to_csv(DATA_CLUSTERING / "evaluacion_k_por_rama.csv", index=False)
evaluacion_por_rama.round(3)


**Decisión de K por rama:**

- **ADMINISTRATIVO (n=693): K=4.** Estabilidad alta y pareja entre K=3 y K=4 (ARI≈0.98-0.99); se
  prefiere K=4 sobre K=3 porque separa con claridad un sub-perfil de **alta movilidad de rol**
  (53 personas, mediana 8 transiciones, todas con transición admin→docente) que en K=3 queda
  mezclado con el grupo de mayor experiencia general. Tamaños: 37-333 personas (5.3%-48.0% de la
  rama) — el grupo de 37 es pequeño pero sustantivo (vinculación a investigación, ver más abajo).
- **DOCENTE (n=1515): K=5.** Estabilidad alta entre K=3 y K=6 (ARI≈0.98); K=5 preserva mejor que
  K=4 un sub-perfil propio de **alta movilidad de rol** (161 personas: máxima diversidad de cargos/
  dedicaciones/categorías de rol, transición docente→admin), que en K=4 se diluye dentro de un grupo
  de alta carga docente sin la señal de movilidad diferenciada. Tamaños: 161-493 (10.6%-32.5%).

En ambas ramas, igual que en la sección 6, no se elige K únicamente por el mejor Silhouette (que
favorecería K=2 en ambos casos) sino por el balance entre estabilidad, tamaños razonables e
interpretabilidad — en particular, la capacidad de aislar el patrón de movilidad de rol que motivó
usar trayectoria como feature (DEC-006).


In [ ]:
K_POR_RAMA = {"ADMINISTRATIVO": 4, "DOCENTE": 5}
# CLUSTER global: ADMINISTRATIVO ocupa 0..K_ADMIN-1, DOCENTE sigue a continuacion.
OFFSET_RAMA = {"ADMINISTRATIVO": 0, "DOCENTE": K_POR_RAMA["ADMINISTRATIVO"]}

modelos_por_rama = {}
subcluster_labels = np.full(len(personas), -1, dtype=int)
cluster_global = np.full(len(personas), -1, dtype=int)

for rama, k in K_POR_RAMA.items():
    mask = grupo_principal == rama
    modelo = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = modelo.fit_predict(X[mask])
    modelos_por_rama[rama] = modelo
    subcluster_labels[mask] = labels
    cluster_global[mask] = labels + OFFSET_RAMA[rama]

# Las 5 personas GRUPO_PRINCIPAL="DESCONOCIDO" (ver celda anterior) quedan intencionalmente en
# -1 (sin sub-perfil): no pertenecen a ninguna rama real, no se les inventa una.
n_desconocido = int((grupo_principal == "DESCONOCIDO").sum())
assert (subcluster_labels >= 0).sum() == len(personas) - n_desconocido
assert (cluster_global >= 0).sum() == len(personas) - n_desconocido

clusters_personas = personas[["IDPERSONA"]].copy()
clusters_personas["GRUPO_PRINCIPAL"] = grupo_principal
clusters_personas["SUBCLUSTER"] = subcluster_labels
clusters_personas["CLUSTER"] = cluster_global

assert clusters_personas["IDPERSONA"].is_unique
clusters_personas.to_csv(DATA_CLUSTERING / "clusters_personas.csv", index=False)

joblib.dump(modelos_por_rama, DATA_CLUSTERING / "modelo_clustering.joblib")

K_FINAL = sum(K_POR_RAMA.values())
print(f"K_FINAL (total de sub-perfiles) = {K_FINAL} ({K_POR_RAMA}); {n_desconocido} personas sin rama (CLUSTER=-1)")
print(clusters_personas.groupby(["GRUPO_PRINCIPAL", "SUBCLUSTER", "CLUSTER"]).size())


In [ ]:
# data/clustering/cluster_sizes.csv (sobreescribe la version de la seccion 6 con la de 2 niveles)
cluster_sizes = (
    clusters_personas.groupby(["CLUSTER", "GRUPO_PRINCIPAL", "SUBCLUSTER"]).size()
    .rename("N_PERSONAS").reset_index()
    .sort_values("CLUSTER").reset_index(drop=True)
)
cluster_sizes["PCT_POBLACION"] = (100 * cluster_sizes["N_PERSONAS"] / len(clusters_personas)).round(2)
cluster_sizes.to_csv(DATA_CLUSTERING / "cluster_sizes.csv", index=False)
cluster_sizes

# data/clustering/clustering_metrics.csv (una fila por rama, en vez de un unico modelo global)
metrics_por_rama = []
for rama, k in K_POR_RAMA.items():
    mask = grupo_principal == rama
    labels_rama = subcluster_labels[mask]
    ari_final = evaluacion_por_rama.loc[
        (evaluacion_por_rama["GRUPO_PRINCIPAL"] == rama) & (evaluacion_por_rama["K"] == k), "ARI_ESTABILIDAD"
    ].iloc[0]
    metrics_por_rama.append({
        "GRUPO_PRINCIPAL": rama, "MODELO": "KMeans", "K": k, "N_PERSONAS": int(mask.sum()),
        "N_FEATURES": X.shape[1],
        "SILHOUETTE": silhouette_score(X[mask], labels_rama),
        "DAVIES_BOULDIN": davies_bouldin_score(X[mask], labels_rama),
        "ARI_ESTABILIDAD_SEMILLAS": ari_final,
        "RANDOM_STATE": RANDOM_STATE, "N_INIT": 10,
    })
clustering_metrics = pd.DataFrame(metrics_por_rama)
clustering_metrics.to_csv(DATA_CLUSTERING / "clustering_metrics.csv", index=False)
clustering_metrics


In [ ]:
# Re-caracterizacion (misma logica de la seccion 7, aplicada al CLUSTER de 2 niveles: 0-3 =
# sub-perfiles de ADMINISTRATIVO, 4-8 = sub-perfiles de DOCENTE). Sobreescribe
# cluster_characterization.csv y cluster_profiles.csv de la seccion 7.
# CLUSTER=-1 (5 personas sin TIPOEMPLEADO_ACTUAL_DESC) se excluye: no pertenecen a un sub-perfil.
df_carac = clusters_personas[clusters_personas["CLUSTER"] != -1].merge(feat_orig, on="IDPERSONA", how="left")
assert df_carac["CLUSTER"].isnull().sum() == 0
clusters_ordenados = sorted(df_carac["CLUSTER"].unique())

filas_caract = []
for feature in features_a_caracterizar:
    tipo = tipo_por_feature.get(feature, "numerica")
    col = df_carac[feature]

    if tipo == "numerica":
        global_val = col.median()
        global_std = col.std()
        for c in clusters_ordenados:
            val_c = col[df_carac["CLUSTER"] == c].median()
            diff = val_c - global_val
            importance = abs(diff) / global_std if global_std and global_std > 0 else 0.0
            signo = ("muy por encima" if importance >= 0.8 else "por encima") if diff > 0 else \
                    ("muy por debajo" if importance >= 0.8 else "por debajo") if diff < 0 else "en línea con"
            interpretacion = f"Mediana de {feature} {signo} del promedio institucional ({val_c:g} vs {global_val:g})."
            filas_caract.append(dict(CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                                      DIFFERENCE=diff, IMPORTANCE=importance, INTERPRETACION=interpretacion))
    elif tipo == "booleana":
        col_bool = col.astype("boolean")
        global_val = col_bool.mean(skipna=True)
        for c in clusters_ordenados:
            val_c = col_bool[df_carac["CLUSTER"] == c].mean(skipna=True)
            diff = val_c - global_val
            interpretacion = f"{val_c:.0%} del cluster cumple {feature}, frente a {global_val:.0%} a nivel global."
            filas_caract.append(dict(CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                                      DIFFERENCE=diff, IMPORTANCE=abs(diff), INTERPRETACION=interpretacion))
    elif tipo in ("categorica", "categorica_ordinal"):
        col_str = col.astype("string").fillna("SIN_DATO")
        vc_global = col_str.value_counts(normalize=True)
        global_mode, global_val = vc_global.index[0], vc_global.iloc[0]
        for c in clusters_ordenados:
            sub = col_str[df_carac["CLUSTER"] == c]
            vc_c = sub.value_counts(normalize=True)
            cluster_mode, cluster_mode_prop = vc_c.index[0], vc_c.iloc[0]
            val_c = vc_c.get(global_mode, 0.0)
            diff = val_c - global_val
            interpretacion = (f"Categoría predominante en el cluster: '{cluster_mode}' ({cluster_mode_prop:.0%}); "
                               f"a nivel global la más común es '{global_mode}' ({global_val:.0%}).")
            filas_caract.append(dict(CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                                      DIFFERENCE=diff, IMPORTANCE=abs(diff), INTERPRETACION=interpretacion))

cluster_characterization = (
    pd.DataFrame(filas_caract).sort_values(["CLUSTER", "IMPORTANCE"], ascending=[True, False]).reset_index(drop=True)
)
cluster_characterization.to_csv(DATA_CLUSTERING / "cluster_characterization.csv", index=False)

perfiles = []
n_total = len(df_carac)
for c in clusters_ordenados:
    sub = df_carac[df_carac["CLUSTER"] == c]
    fila = {"CLUSTER": c, "N_PERSONAS": len(sub), "PCT_POBLACION": round(100 * len(sub) / n_total, 2)}
    for f in numericas:
        fila[f"{f}__MEDIA"] = sub[f].mean()
        fila[f"{f}__MEDIANA"] = sub[f].median()
    for f in booleanas:
        fila[f"{f}__PROPORCION"] = sub[f].astype("boolean").mean(skipna=True)
    for f in categoricas:
        vc = sub[f].astype("string").fillna("SIN_DATO").value_counts(normalize=True)
        fila[f"{f}__MODA"] = vc.index[0]
        fila[f"{f}__MODA_PROP"] = vc.iloc[0]
    perfiles.append(fila)

cluster_profiles = pd.DataFrame(perfiles)
cluster_profiles.to_csv(DATA_CLUSTERING / "cluster_profiles.csv", index=False)
print(cluster_characterization.shape, cluster_profiles.shape)

for c in clusters_ordenados:
    fila_size = cluster_sizes[cluster_sizes.CLUSTER == c].iloc[0]
    print(f"\n=== CLUSTER {c} ({fila_size['GRUPO_PRINCIPAL']}, n={fila_size['N_PERSONAS']}) ===")
    for _, r in cluster_characterization[cluster_characterization["CLUSTER"] == c].head(6).iterrows():
        print(f"  - {r['INTERPRETACION']}")


### Nombres de los 9 sub-perfiles (4 administrativos + 5 docentes)

Igual que en la sección 8, los nombres se proponen después de revisar la evidencia (variables más
diferenciadoras impresa arriba), y no son una categoría institucional oficial.

**Administrativo (4 sub-perfiles):**
- **0** (270, 12.2% de la población): sin títulos de posgrado, bajo número de regímenes distintos,
  perfil de entrada estándar. → **"Administrativo de ingreso reciente"**.
- **1** (53, 2.4%): el más regímenes/cargos distintos (hasta 8 cargos, 3 regímenes), mayor
  `N_CONTRATOS_TOTAL` — la señal de trayectoria más fuerte de toda la rama administrativa: 100% tuvo
  alguna transición administrativo→docente, mediana 8 transiciones. → **"Administrativo con alta
  movilidad hacia docencia"**.
- **2** (37, 1.7%): el único sub-perfil administrativo con actividad de investigación relevante
  (mediana 11 proyectos, participación en publicaciones Q1) pese a estar tipificado como
  administrativo. → **"Administrativo vinculado a investigación"**.
- **3** (333, 15.0%): una sola unidad organizacional, alta heteroevaluación histórica, bajo número
  de registros nuevos — el grupo administrativo más numeroso y con trayectoria más consolidada.
  → **"Administrativo de trayectoria consolidada"**.

**Docente (5 sub-perfiles):**
- **4** (493, 22.3%): antigüedad mediana de 0.85 años, cero periodos de docencia — el ingreso más
  reciente de toda la población. → **"Docente de ingreso muy reciente"**.
- **5** (161, 7.3%): máxima diversidad de cargos (11 distintos) y dedicaciones (3), mayor
  `N_CONTRATOS_TOTAL` de la rama docente — la contraparte docente del sub-perfil 1: alguna vez
  transicionó desde administrativo. → **"Docente con alta movilidad de rol"**.
- **6** (219, 9.9%): máxima actividad de investigación (mediana 12 proyectos, participación como
  director). → **"Docente de alta producción académica e investigativa"**.
- **7** (392, 17.7%): sin investigación, cobertura de evaluación baja, antigüedad corta (~3.4 años)
  — un ingreso algo más consolidado que el sub-perfil 4 pero sin producción relevante todavía.
  → **"Docente ocasional de corta duración"**.
- **8** (250, 11.3%): máxima carga de docencia pura (70 cursos, 5197 horas, 1440 estudiantes).
  → **"Docente de alta carga docente"**.

**Advertencia importante (igual que en la sección 8):** estos nombres son una síntesis
interpretativa, no una categoría institucional oficial de ESPOL, y no deben usarse para
automatizar decisiones de contratación, promoción o asignación sin criterio institucional.


In [ ]:
nombres_perfiles = {
    0: "Administrativo de ingreso reciente",
    1: "Administrativo con alta movilidad hacia docencia",
    2: "Administrativo vinculado a investigación",
    3: "Administrativo de trayectoria consolidada",
    4: "Docente de ingreso muy reciente",
    5: "Docente con alta movilidad de rol",
    6: "Docente de alta producción académica e investigativa",
    7: "Docente ocasional de corta duración",
    8: "Docente de alta carga docente",
}

nombres_df = pd.DataFrame({"CLUSTER": list(nombres_perfiles.keys()), "NOMBRE_PERFIL": list(nombres_perfiles.values())})
cluster_sizes_nombrado = cluster_sizes.merge(nombres_df, on="CLUSTER").sort_values("CLUSTER")
cluster_sizes_nombrado.to_csv(DATA_CLUSTERING / "cluster_sizes.csv", index=False)  # incluye NOMBRE_PERFIL
cluster_sizes_nombrado


### Proyección 2D para el dashboard (clic por persona) y dendrograma de los 9 sub-perfiles

Se guarda una proyección PCA de 2 componentes (`data/clustering/pca_personas.csv`, `IDPERSONA` +
`PC1` + `PC2`), calculada una sola vez sobre las 141 columnas completas de `X_modelado` (mismo PCA
para ambas ramas, para que las distancias en el gráfico sean comparables entre administrativos y
docentes), pensada para que el dashboard dibuje un punto por persona, coloreado por sub-perfil y
con clic para identificar quién es — sin tener que cargar `X_modelado.csv` completo desde Streamlit.

También se arma el dendrograma (ward) de los 9 centroides (uno por sub-perfil, tomados de cada
modelo de rama) para ver qué tan parecidos son entre sí, ahora dentro de la estructura de 2 niveles.


In [ ]:
# PCA 2D sobre X completo, guardado por persona para el dashboard
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
var_explicada = pca.explained_variance_ratio_

pca_personas = personas[["IDPERSONA"]].copy()
pca_personas["PC1"] = X_pca[:, 0]
pca_personas["PC2"] = X_pca[:, 1]
pca_personas.to_csv(DATA_CLUSTERING / "pca_personas.csv", index=False)
print(f"Varianza explicada por PC1+PC2: {var_explicada.sum():.1%} (baja cobertura esperable en 141 dimensiones)")

fig, ax = plt.subplots(figsize=(9, 7))
paleta = sns.color_palette("tab10", K_FINAL)
for c in clusters_ordenados:
    mask_c = cluster_global == c
    ax.scatter(X_pca[mask_c, 0], X_pca[mask_c, 1], s=12, alpha=0.6, color=paleta[c], label=f"{c}: {nombres_perfiles[c]}")
ax.set_xlabel(f"PC1 ({var_explicada[0]:.1%})")
ax.set_ylabel(f"PC2 ({var_explicada[1]:.1%})")
ax.set_title("9 sub-perfiles (2 niveles: AA/DD) proyectados en 2D con PCA — solo visualización")
ax.legend(markerscale=2, fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

# Dendrograma de los 9 centroides (uno por rama x sub-perfil)
centroides_9 = np.zeros((K_FINAL, X.shape[1]))
for rama, k in K_POR_RAMA.items():
    centros = modelos_por_rama[rama].cluster_centers_
    for i in range(k):
        centroides_9[i + OFFSET_RAMA[rama]] = centros[i]

etiquetas_centroides = [f"{c}: {nombres_perfiles[c]}\n(n={cluster_sizes_nombrado.loc[cluster_sizes_nombrado.CLUSTER==c, 'N_PERSONAS'].iloc[0]})"
                         for c in range(K_FINAL)]
Z_centroides = linkage(centroides_9, method="ward")

fig, ax = plt.subplots(figsize=(11, 5))
dendrogram(Z_centroides, labels=etiquetas_centroides, ax=ax, leaf_rotation=90, leaf_font_size=8)
ax.set_title(f"Dendrograma de los {K_FINAL} sub-perfiles (centroides, ward, espacio de X_modelado)")
ax.set_ylabel("distancia (ward) entre centroides")
plt.tight_layout()
plt.show()

dendrograma_centroides = pd.DataFrame(Z_centroides, columns=["CLUSTER_A", "CLUSTER_B", "DISTANCIA", "N_OBSERVACIONES"])
dendrograma_centroides.to_csv(DATA_CLUSTERING / "dendrograma_centroides.csv", index=False)
dendrograma_centroides


## 12. Resumen — estructura final vigente (DEC-008)

**Decisión tomada:** clustering en **2 niveles**. Nivel 1 = `TIPOEMPLEADO_ACTUAL_DESC`
(Administrativo/Docente, dato institucional, no descubierto). Nivel 2 = K-Means independiente por
rama (K=4 en Administrativo, K=5 en Docente), sobre las mismas 141 columnas de `X_modelado.csv`,
`random_state=42`, `n_init=10`. Registrado como **DEC-008** en `context/DECISION_LOG.md`, reemplaza
a DEC-007 (K=5 global) como estructura final — DEC-007 sigue documentado en las secciones 1-10 como
parte del proceso exploratorio.

**Archivos en `data/clustering/` (sobreescritos con la estructura de 2 niveles):**

| Archivo | Contenido |
|---|---|
| `clusters_personas.csv` | `IDPERSONA`, `GRUPO_PRINCIPAL`, `SUBCLUSTER` (0-indexado dentro de la rama), `CLUSTER` (id global 0-8) |
| `cluster_sizes.csv` | Tamaño, % y `NOMBRE_PERFIL` por `CLUSTER` |
| `cluster_profiles.csv` / `cluster_characterization.csv` | Igual que antes, recalculados sobre los 9 sub-perfiles |
| `clustering_metrics.csv` | Una fila por rama (Administrativo, Docente) con sus propias métricas |
| `modelo_clustering.joblib` | Diccionario `{"ADMINISTRATIVO": KMeans, "DOCENTE": KMeans}` |
| `evaluacion_k_por_rama.csv` | Evaluación de K=2..8 dentro de cada rama |
| `pca_personas.csv` | `IDPERSONA`, `PC1`, `PC2` — para el gráfico de puntos clicable del dashboard |
| `dendrograma_centroides.csv` | Enlace ward entre los 9 centroides |

**Limitaciones:** las mismas de la sección 10 (naturaleza mixta de `X_modelado`, PCA como ayuda
visual únicamente, patrón de "encargos cortos" de DEC-004 sin resolver), más una nueva: al
clusterizar cada rama por separado, la escala/varianza usada por K-Means ya no es exactamente la
misma que si se clusterizara junto (los `StandardScaler` de la etapa 05 se ajustaron sobre toda la
población, no por rama) — no se reajustó el escalamiento por rama para mantener comparabilidad con
`X_modelado.csv` tal como lo consume el resto del proyecto (embeddings, dashboard).

**Siguiente paso:** actualizar `notebooks/08_dashboard/lib.py` (`PERFIL_NOMBRES` con las 9
entradas y su `GRUPO_PRINCIPAL`) y `app.py` (gráfico de puntos interactivo con `pca_personas.csv`,
filtro Administrativo/Docente, clic para identificar persona), y regenerar `data/dashboard/`.


## 13. Jerarquia por cargo real (no descubierta): TIPOEMPLEADO -> CATEGORIA_CARGO_ACTUAL (DEC-009)

> **2026-09-08 (DEC-009, reemplaza DEC-008 como estructura final):** el usuario corrigio el
> diseno de la seccion 11-12: agrupar por K-Means dentro de cada rama (administrativo/docente)
> convertia patrones de actividad (movilidad de rol, produccion, etc.) en **categorias de
> grupo**, cuando esa senal ya vive en las variables y debe verse como **cercania en el mapa**,
> no como una etiqueta nueva. Un docente cuya trayectoria se parece a la de un subdecano debe
> seguir etiquetado como lo que realmente es (su cargo real), pero puede aparecer *cerca* del
> grupo de autoridades en el grafico de puntos (seccion 13.3) - eso es lo que permite explorar
> afinidad de trayectoria, no una categoria forzada.

**Diseno correcto:** el nivel 2 de la jerarquia no se descubre con clustering, se toma
directamente de `CATEGORIA_CARGO_ACTUAL` (`04_trayectorias.ipynb`, DEC-004): la categoria de
cargo del tramo de rol vigente de cada persona (Autoridad Academica, Docente Titular, Tecnico
Docente, Profesional/Analista, etc.). No hay K a elegir ni estabilidad que evaluar - es una
asignacion basada en reglas ya validadas con el usuario, no un resultado de K-Means.

**Confirmado con el usuario:** Rector/Vicerrector/Decano/Subdecano tienen `TIPOEMPLEADO=DOCENTE`
en el dato institucional (se eligen desde el escalafon docente) y se mantienen ahi -
`AUTORIDAD_ACADEMICA_SUPERIOR` es una subcategoria de Docente, no de Administrativo, aunque
funcionalmente ejerzan un cargo directivo.


> **Actualizado 2026-09-09 (DEC-010):** se corrigio una inconsistencia real encontrada por el usuario en el dashboard (categorias como "Autoridad administrativa (Gerente/Director)" mostraban un pequeno % de personas etiquetadas como Docente). Causa raiz: GRUPO_PRINCIPAL se tomaba de TIPOEMPLEADO_ACTUAL_DESC (dataset_personas_features.csv, calculado sobre el ultimo contrato por fecha), una fuente de "actual" distinta a la que produce CATEGORIA_CARGO_ACTUAL (el tramo vigente no puntual de tramos_rol.csv). Se corrigio en el origen: construir_features_trayectoria ahora tambien devuelve TIPOEMPLEADO_CATEGORIA_ACTUAL, el TIPOEMPLEADO_DESC del *mismo tramo* usado para CATEGORIA_CARGO_ACTUAL - y es esa columna la que se usa como GRUPO_PRINCIPAL. Por construccion, ya no puede haber una categoria con personas de las dos ramas.

In [ ]:
# GRUPO_PRINCIPAL (nivel 1) + CATEGORIA_CARGO_ACTUAL (nivel 2, regla ya validada en
# 04_trayectorias.ipynb) - DEC-010: GRUPO_PRINCIPAL ahora se toma de
# TIPOEMPLEADO_CATEGORIA_ACTUAL (el TIPOEMPLEADO_DESC del *mismo tramo* que produjo
# CATEGORIA_CARGO_ACTUAL, ver construir_features_trayectoria), NO de la columna
# TIPOEMPLEADO_ACTUAL_DESC de dataset_personas_features.csv (calculada por separado, sobre
# el ultimo contrato por fecha). Antes (DEC-009) ambas podian no coincidir en rama -p.ej.
# alguien "Autoridad administrativa (Gerente/Director)" apareciendo como Docente- porque
# comparaban dos fuentes de "actual" distintas. Al usar el TIPOEMPLEADO del mismo tramo,
# GRUPO_PRINCIPAL y CATEGORIA_CARGO_ACTUAL quedan consistentes por construccion: cada
# categoria ya es exclusiva de una rama (clasificar_categoria_cargo nunca comparte reglas
# entre AA y DD), asi que ya no puede haber una persona "Gerente" etiquetada como Docente.
df_jerarquia = personas[["IDPERSONA"]].merge(
    feat_orig[["IDPERSONA", "TIPOEMPLEADO_CATEGORIA_ACTUAL", "CATEGORIA_CARGO_ACTUAL"]],
    on="IDPERSONA", how="left",
).rename(columns={"TIPOEMPLEADO_CATEGORIA_ACTUAL": "TIPOEMPLEADO_ACTUAL_DESC"})

sin_categoria = df_jerarquia["CATEGORIA_CARGO_ACTUAL"].isna()
print(f"Personas sin CATEGORIA_CARGO_ACTUAL (sin tramo de rol, ver DEC-004): {sin_categoria.sum()}")

tabla_cruce = pd.crosstab(df_jerarquia["TIPOEMPLEADO_ACTUAL_DESC"], df_jerarquia["CATEGORIA_CARGO_ACTUAL"])
n_discrepancias = int((tabla_cruce > 0).sum(axis=0).gt(1).sum())
print(f"\nCategorias con mas de una rama distinta (deberia ser 0 tras DEC-010): {n_discrepancias}")
print(tabla_cruce)


### 13.1 Nombres legibles de cada categoria

Mismos nombres de categoria que en `04_trayectorias.ipynb`/`_preprocesamiento_comun.py`, en
version legible para el dashboard.


In [ ]:
NOMBRES_CATEGORIA_CARGO = {
    "AUTORIDAD_ACADEMICA_SUPERIOR": "Autoridad académica (Rector/Vicerrector/Decano/Subdecano)",
    "DOCENTE_TITULAR_CARRERA": "Docente titular de carrera",
    "DOCENTE_NO_TITULAR_OCASIONAL": "Docente no titular / ocasional",
    "DOCENTE_CONTRATADO_SERVICIOS_CIVILES": "Docente contratado (servicios civiles)",
    "DOCENTE_HONORARIO_ESPECIAL": "Docente honorario / invitado",
    "DOCENTE_PREPOLITECNICO": "Docente pre-politécnico",
    "DOCENTE_INVESTIGADOR": "Docente investigador",
    "DIRECCION_ACADEMICA_INTERMEDIA": "Dirección académica intermedia",
    "AYUDANTE_ACADEMICO_JUNIOR": "Ayudante académico junior",
    "TECNICO_DOCENTE_APOYO": "Técnico docente / de investigación",
    "AUTORIDAD_ADMINISTRATIVA_SUPERIOR": "Autoridad administrativa (Gerente/Director)",
    "JEFATURA_SUPERVISION_OPERATIVA": "Jefatura / supervisión operativa",
    "PROFESIONAL_ANALISTA": "Profesional / Analista",
    "PROFESIONAL_ESPECIALIZADO_TECNICO": "Profesional especializado (abogado, psicólogo, etc.)",
    "ASISTENCIA_SECRETARIAL_OFICINA": "Asistencia secretarial / oficina",
    "AUXILIAR_AYUDANTE_OPERATIVO": "Auxiliar / ayudante operativo",
    "TECNICO_OPERATIVO_MANTENIMIENTO": "Técnico operativo / mantenimiento",
    "SERVICIOS_GENERALES_OFICIOS": "Servicios generales / oficios",
    "CONTRATO_SERVICIOS_PROFESIONALES_PROYECTO": "Contrato de servicios profesionales (proyecto)",
    "TRIBUNAL_COMISION_ACADEMICA": "Tribunal / comisión académica",
    "ACTIVIDAD_ACADEMICA_DESDE_ADMINISTRATIVO": "Actividad académica desde cargo administrativo",
    "OTROS_AA": "Otros (administrativo)",
    "OTROS_DD": "Otros (docente)",
}

categorias_presentes = sorted(df_jerarquia["CATEGORIA_CARGO_ACTUAL"].dropna().unique())
sin_nombre = [c for c in categorias_presentes if c not in NOMBRES_CATEGORIA_CARGO]
assert not sin_nombre, f"Categorias sin nombre legible definido: {sin_nombre}"
print(f"{len(categorias_presentes)} categorias presentes en esta poblacion (de {len(NOMBRES_CATEGORIA_CARGO)} posibles).")


### 13.2 Estructura final: `clusters_personas.csv`, tamaños y caracterización

`CLUSTER` (id numérico estable) se asigna por orden alfabético de `CATEGORIA_CARGO_ACTUAL` —
ya no representa un resultado de K-Means, es una tabla de traducción categoría→id para que el
resto del pipeline (dashboard, `lib.PERFIL_NOMBRES`) siga funcionando igual que antes.


In [ ]:
categoria_a_id = {cat: i for i, cat in enumerate(categorias_presentes)}

clusters_personas = df_jerarquia.rename(columns={
    "TIPOEMPLEADO_ACTUAL_DESC": "GRUPO_PRINCIPAL",
    "CATEGORIA_CARGO_ACTUAL": "SUBGRUPO",
}).copy()
clusters_personas["CLUSTER"] = clusters_personas["SUBGRUPO"].map(categoria_a_id)
clusters_personas["CLUSTER"] = clusters_personas["CLUSTER"].fillna(-1).astype(int)

assert clusters_personas["IDPERSONA"].is_unique
clusters_personas.to_csv(DATA_CLUSTERING / "clusters_personas.csv", index=False)

nombres_perfiles = {i: NOMBRES_CATEGORIA_CARGO[cat] for cat, i in categoria_a_id.items()}
K_FINAL = len(categorias_presentes)

cluster_sizes = (
    clusters_personas[clusters_personas["CLUSTER"] != -1]
    .groupby(["CLUSTER", "GRUPO_PRINCIPAL", "SUBGRUPO"]).size()
    .rename("N_PERSONAS").reset_index().sort_values("CLUSTER").reset_index(drop=True)
)
cluster_sizes["PCT_POBLACION"] = (100 * cluster_sizes["N_PERSONAS"] / len(clusters_personas)).round(2)
cluster_sizes["NOMBRE_PERFIL"] = cluster_sizes["CLUSTER"].map(nombres_perfiles)
cluster_sizes.to_csv(DATA_CLUSTERING / "cluster_sizes.csv", index=False)
print(f"K_FINAL (categorias de cargo con al menos 1 persona) = {K_FINAL}")
cluster_sizes


### 12b. Detalle de personas sin cluster (`CLUSTER=-1`, DEC-018)

Investigacion pedida por el usuario: por que 719 personas quedan sin cluster. La causa es siempre la misma condicion documentada desde DEC-004/DEC-012 - no tienen ningun tramo de rol ESTRUCTURAL (`CATEGORIA_CARGO_ACTUAL` nulo en `features_trayectoria_persona.csv`) porque TODO su historial laboral (excluyendo `ES_RUIDO_CALIDAD_DATOS`) cae en categorias puntuales/sin relacion de dependencia (`CATEGORIAS_PUNTUALES`, `SIN_TIPOEMPLEADO`, `SIN_DATO`), o no tienen ningun registro en `historial_laboral_personas.csv`. No es un error de la implementacion: no se fuerza una asignacion de cluster porque no corresponde tecnicamente (no hay un cargo estructural real que perfilar). Se guarda el detalle completo para que pueda revisarse caso por caso.

In [ ]:
hist_cat_cargo = pd.read_csv(ROOT / "data" / "trayectorias" / "historial_laboral_categoria_cargo.csv", low_memory=False)
for _c in ("FECHAINICIOCONTRATO", "FECHAFINCONTRATO"):
    hist_cat_cargo[_c] = pd.to_datetime(hist_cat_cargo[_c], format="mixed", errors="coerce")

contrato_puntual_vigente = pd.read_csv(
    ROOT / "data" / "trayectorias" / "features_trayectoria_persona.csv",
    usecols=["IDPERSONA", "CARGO_PUNTUAL_VIGENTE"],
)
fallback_map = contrato_puntual_vigente.set_index("IDPERSONA")["CARGO_PUNTUAL_VIGENTE"].dropna()

sin_cluster_ids = clusters_personas.loc[clusters_personas["CLUSTER"] == -1, "IDPERSONA"]

def _motivo_fila(idp):
    sub = hist_cat_cargo[(hist_cat_cargo["IDPERSONA"] == idp) & (~hist_cat_cargo["ES_RUIDO_CALIDAD_DATOS"])]
    sub = sub.sort_values("FECHAINICIOCONTRATO")
    if sub.empty:
        return pd.Series({"MOTIVO_CATEGORIA": "SIN_HISTORIAL_LABORAL", "ULTIMO_CARGO_REGISTRADO": pd.NA,
                           "FECHA_INICIO_ULTIMO_CONTRATO": pd.NaT})
    ultima = sub.iloc[-1]
    return pd.Series({"MOTIVO_CATEGORIA": ultima["CATEGORIA_CARGO"], "ULTIMO_CARGO_REGISTRADO": ultima["CARGO"],
                       "FECHA_INICIO_ULTIMO_CONTRATO": ultima["FECHAINICIOCONTRATO"]})

personas_sin_cluster = pd.DataFrame({"IDPERSONA": sin_cluster_ids}).reset_index(drop=True)
personas_sin_cluster = personas_sin_cluster.join(personas_sin_cluster["IDPERSONA"].apply(_motivo_fila))
personas_sin_cluster["TIENE_CONTRATO_PUNTUAL_VIGENTE_HOY"] = personas_sin_cluster["IDPERSONA"].isin(fallback_map.index)
personas_sin_cluster = personas_sin_cluster.sort_values(["MOTIVO_CATEGORIA", "IDPERSONA"]).reset_index(drop=True)

personas_sin_cluster.to_csv(DATA_CLUSTERING / "personas_sin_cluster_detalle.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: {DATA_CLUSTERING / 'personas_sin_cluster_detalle.csv'} ({len(personas_sin_cluster)} filas)")
print()
print("Motivo (categoria del ultimo contrato no-ruido):")
print(personas_sin_cluster["MOTIVO_CATEGORIA"].value_counts().to_string())
personas_sin_cluster.head(10)

In [ ]:
# Re-caracterizacion (misma logica de las secciones 7/11), ahora sobre las categorias de cargo
# reales en vez de sub-perfiles de K-Means. CLUSTER=-1 (5 personas sin tramo de rol) se excluye.
df_carac = clusters_personas[clusters_personas["CLUSTER"] != -1].merge(feat_orig, on="IDPERSONA", how="left")
assert df_carac["CLUSTER"].isnull().sum() == 0
clusters_ordenados = sorted(df_carac["CLUSTER"].unique())

filas_caract = []
for feature in features_a_caracterizar:
    tipo = tipo_por_feature.get(feature, "numerica")
    col = df_carac[feature]
    if tipo == "numerica":
        global_val = col.median(); global_std = col.std()
        for c in clusters_ordenados:
            val_c = col[df_carac["CLUSTER"] == c].median()
            diff = val_c - global_val
            importance = abs(diff) / global_std if global_std and global_std > 0 else 0.0
            signo = ("muy por encima" if importance >= 0.8 else "por encima") if diff > 0 else \
                    ("muy por debajo" if importance >= 0.8 else "por debajo") if diff < 0 else "en línea con"
            interpretacion = f"Mediana de {feature} {signo} del promedio institucional ({val_c:g} vs {global_val:g})."
            filas_caract.append(dict(CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                                      DIFFERENCE=diff, IMPORTANCE=importance, INTERPRETACION=interpretacion))
    elif tipo == "booleana":
        col_bool = col.astype("boolean"); global_val = col_bool.mean(skipna=True)
        for c in clusters_ordenados:
            val_c = col_bool[df_carac["CLUSTER"] == c].mean(skipna=True)
            diff = val_c - global_val
            interpretacion = f"{val_c:.0%} del cluster cumple {feature}, frente a {global_val:.0%} a nivel global."
            filas_caract.append(dict(CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                                      DIFFERENCE=diff, IMPORTANCE=abs(diff), INTERPRETACION=interpretacion))
    elif tipo in ("categorica", "categorica_ordinal"):
        col_str = col.astype("string").fillna("SIN_DATO")
        vc_global = col_str.value_counts(normalize=True)
        global_mode, global_val = vc_global.index[0], vc_global.iloc[0]
        for c in clusters_ordenados:
            sub = col_str[df_carac["CLUSTER"] == c]
            vc_c = sub.value_counts(normalize=True)
            cluster_mode, cluster_mode_prop = vc_c.index[0], vc_c.iloc[0]
            val_c = vc_c.get(global_mode, 0.0)
            diff = val_c - global_val
            interpretacion = (f"Categoría predominante en el cluster: '{cluster_mode}' ({cluster_mode_prop:.0%}); "
                               f"a nivel global la más común es '{global_mode}' ({global_val:.0%}).")
            filas_caract.append(dict(CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                                      DIFFERENCE=diff, IMPORTANCE=abs(diff), INTERPRETACION=interpretacion))

cluster_characterization = (
    pd.DataFrame(filas_caract).sort_values(["CLUSTER", "IMPORTANCE"], ascending=[True, False]).reset_index(drop=True)
)
cluster_characterization.to_csv(DATA_CLUSTERING / "cluster_characterization.csv", index=False)

perfiles = []
n_total = len(df_carac)
for c in clusters_ordenados:
    sub = df_carac[df_carac["CLUSTER"] == c]
    fila = {"CLUSTER": c, "N_PERSONAS": len(sub), "PCT_POBLACION": round(100 * len(sub) / n_total, 2)}
    for f in numericas:
        fila[f"{f}__MEDIA"] = sub[f].mean(); fila[f"{f}__MEDIANA"] = sub[f].median()
    for f in booleanas:
        fila[f"{f}__PROPORCION"] = sub[f].astype("boolean").mean(skipna=True)
    for f in categoricas:
        vc = sub[f].astype("string").fillna("SIN_DATO").value_counts(normalize=True)
        fila[f"{f}__MODA"] = vc.index[0]; fila[f"{f}__MODA_PROP"] = vc.iloc[0]
    perfiles.append(fila)

cluster_profiles = pd.DataFrame(perfiles)
cluster_profiles.to_csv(DATA_CLUSTERING / "cluster_profiles.csv", index=False)
print(cluster_characterization.shape, cluster_profiles.shape)


### 13.3 Mapa de puntos (afinidad de trayectoria) y dendrograma de categorías

El PCA es el mismo cálculo de la sección 11-12 (no depende de cómo se agrupe después): se
recalcula aquí solo para que esta sección sea autocontenida. Lo importante es que **todas las
personas comparten el mismo espacio 2D**, sin importar su categoría de cargo — así, un docente
cuya trayectoria se parezca a la de las autoridades académicas puede aparecer visualmente cerca
de ese grupo en el mapa, sin que eso cambie su categoría real. Esa cercanía es exactamente la
señal de "afinidad de trayectoria" que pidió el usuario, y **no** se convierte en una nueva
categoría ni en una predicción.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
var_explicada = pca.explained_variance_ratio_

pca_personas = personas[["IDPERSONA"]].copy()
pca_personas["PC1"] = X_pca[:, 0]
pca_personas["PC2"] = X_pca[:, 1]
pca_personas.to_csv(DATA_CLUSTERING / "pca_personas.csv", index=False)
print(f"Varianza explicada por PC1+PC2: {var_explicada.sum():.1%}")

fig, ax = plt.subplots(figsize=(10, 8))
paleta = sns.color_palette("tab20", K_FINAL)
cluster_por_fila = clusters_personas["CLUSTER"].to_numpy()
for c in clusters_ordenados:
    mask_c = cluster_por_fila == c
    ax.scatter(X_pca[mask_c, 0], X_pca[mask_c, 1], s=12, alpha=0.6, color=paleta[c], label=f"{c}: {nombres_perfiles[c]}")
ax.set_xlabel(f"PC1 ({var_explicada[0]:.1%})")
ax.set_ylabel(f"PC2 ({var_explicada[1]:.1%})")
ax.set_title("Categorías de cargo reales proyectadas en 2D — para explorar afinidad de trayectoria")
ax.legend(markerscale=2, fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

# Dendrograma de los centroides de cada categoria real (no de sub-perfiles de K-Means)
centroides = np.array([X[cluster_por_fila == c].mean(axis=0) for c in clusters_ordenados])
etiquetas_centroides = [f"{c}: {nombres_perfiles[c]}\n(n={cluster_sizes.loc[cluster_sizes.CLUSTER==c, 'N_PERSONAS'].iloc[0]})"
                         for c in clusters_ordenados]
Z_centroides = linkage(centroides, method="ward")

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z_centroides, labels=etiquetas_centroides, ax=ax, leaf_rotation=90, leaf_font_size=7)
ax.set_title(f"Dendrograma de las {K_FINAL} categorías de cargo reales (centroides, ward)")
ax.set_ylabel("distancia (ward) entre centroides")
plt.tight_layout()
plt.show()

dendrograma_centroides = pd.DataFrame(Z_centroides, columns=["CLUSTER_A", "CLUSTER_B", "DISTANCIA", "N_OBSERVACIONES"])
dendrograma_centroides.to_csv(DATA_CLUSTERING / "dendrograma_centroides.csv", index=False)


### 13.4 Resumen — estructura final vigente (DEC-009)

**Decisión tomada:** la jerarquía de 2 niveles ya no se descubre con K-Means (DEC-008): el
nivel 2 es `CATEGORIA_CARGO_ACTUAL`, la categoría de cargo real de la persona (regla ya validada
en DEC-004), sin clustering adicional. El mapa de puntos (PCA sobre las 141 columnas de
`X_modelado`) es el mecanismo para explorar *afinidad de trayectoria* por cercanía visual, no
para definir nuevas categorías. Registrado como **DEC-009**, reemplaza a DEC-008.

**Archivos sobreescritos:** `clusters_personas.csv` (`GRUPO_PRINCIPAL`, `SUBGRUPO` =
`CATEGORIA_CARGO_ACTUAL`, `CLUSTER` = id numérico de traducción), `cluster_sizes.csv`,
`cluster_profiles.csv`, `cluster_characterization.csv`, `pca_personas.csv`,
`dendrograma_centroides.csv`.

**Limitación conocida (nueva):** `TIPOEMPLEADO_ACTUAL_DESC` (de `historial_laboral_features`,
último contrato) y `CATEGORIA_CARGO_ACTUAL` (de `tramos_rol`, tramo vigente no puntual) se
calculan con lógicas de "actual" ligeramente distintas y no siempre coinciden en la rama
esperada (ver tabla de cruce en 13.1) — la más notable: 74 personas `TIPOEMPLEADO=ADMINISTRATIVO`
cuya categoría de cargo vigente es `TECNICO_DOCENTE_APOYO`. No se fuerza a coincidir; queda
documentado para una futura revisión de consistencia entre ambas fuentes.

**Pendiente:** actualizar `notebooks/08_dashboard/lib.py` (`PERFIL_NOMBRES` con las categorías
reales) y `app.py` (filtro por categoría de cargo en vez de sub-perfil de K-Means; el mapa de
puntos se mantiene igual, ya estaba pensado para esto) y regenerar `data/dashboard/`.
